# Part 2: Time-Series Foundation Models
## Univariate Baseline with TimesFM 3.0

In this notebook, we establish our zero-shot univariate forecasting baseline using Google's **TimesFM 3.0**. 

As we discussed, we will:
1. Generate the chaotic environment using our `radar_generator`.
2. Simulate a perfect deinterleaver by filtering for a specific Radar ID.
3. Treat the pulses as an **Event Sequence**, using the Pulse Number as the index and `Measured_Delta_TOA` (the PRI) as the numerical target to forecast.

### Prerequisites
We are installing `timesfm[torch]` to ensure we get the PyTorch backend, which is highly compatible with Windows 11 CUDA.

In [ ]:
!pip install timesfm[torch]

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add the synthetic data generation folder to our path so we can import the generator
sys.path.append(os.path.abspath(os.path.join('..', 'synthetic data generation')))
from radar_generator import Environment, StableRadar, JitterRadar, StaggerRadar, SlidingRadar, DwellAndSwitchRadar

plt.style.use('seaborn-v0_8-whitegrid')

### 1. Data Generation & Deinterleaving
We generate the dataset and isolate a complex radar (e.g., Stagger or Dwell-and-Switch) to see if TimesFM can learn its pattern.

In [ ]:
env = Environment()

# Let's use a Stagger Radar with 3 different PRIs
stagger_radar = StaggerRadar(
    radar_id="R_STAGGER", 
    stagger_pris=[800.0, 1200.0, 1500.0], 
    freq_mean=8500.0, 
    poi=0.95, # 5% missing pulses to make it challenging
)
env.add_radar(stagger_radar)

# Generate data
df_interleaved = env.generate(duration=10_000_000.0) # 10 seconds

# "Deinterleave" by filtering
df_target = df_interleaved[df_interleaved["Radar_ID"] == "R_STAGGER"].copy()
df_target = df_target.reset_index(drop=True)

# Extract the target time series (Measured PRI)
pri_series = df_target["Measured_Delta_TOA"].values

print(f"Extracted {len(pri_series)} pulses for R_STAGGER.")

### 2. TimesFM 3.0 Initialization
We load the `google/timesfm-3.0-pytorch` weights from Hugging Face using the new `TimesFM3Forecaster` API. It should automatically detect your CUDA device.

In [ ]:
import torch
try:
    from timesfm import TimesFM3Forecaster
except ImportError:
    # Fallback depending on exactly how the pip package exposed the module
    from timesfm3.pytorch import TimesFM3Forecaster

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Initialize the TimesFM 3.0 forecaster (downloads weights automatically if not cached)
tfm = TimesFM3Forecaster.from_pretrained(
    "google/timesfm-3.0-pytorch",
    per_core_batch_size=16  # Adjust based on VRAM
)

print(f"\nTimesFM3 forecaster loaded successfully on device: {tfm.device}")

### 3. Zero-Shot Forecasting
We give the model a sequence of past pulses (context) and ask it to predict the next `horizon` pulses using the new `.predict()` method.

In [ ]:
context_length = 512
horizon_length = 64

# Our context is the first 512 pulses
context_data = pri_series[:context_length]

# The ground truth is the next 64 pulses
ground_truth = pri_series[context_length : context_length + horizon_length]

# Run inference using the new predict method
# It expects a single 1D numpy array for context and the argument is `horizon`
forecast_output = tfm.predict(context_data, horizon=horizon_length)

# Extract the point predictions array from the ForecastOutput object
predictions = forecast_output.forecast

# Plotting the results
plt.figure(figsize=(15, 6))
x_context = np.arange(context_length - 100, context_length) # Show last 100 context points
x_horizon = np.arange(context_length, context_length + horizon_length)

plt.plot(x_context, context_data[-100:], label="Context (Observed PRI)", color="blue", marker='o', markersize=4)
plt.plot(x_horizon, ground_truth, label="Ground Truth (Actual Future)", color="green", marker='o', markersize=4)
plt.plot(x_horizon, predictions, label="TimesFM 3.0 Forecast", color="red", linestyle="--", marker='x', markersize=6)

plt.axvline(x=context_length - 0.5, color="black", linestyle="--", alpha=0.5, label="Prediction Start")
plt.title("TimesFM 3.0 Zero-Shot PRI Forecasting (R_STAGGER)")
plt.xlabel("Pulse Sequence Index (n)")
plt.ylabel("PRI / Measured Delta TOA (us)")
plt.legend()
plt.tight_layout()
plt.show()